In [ ]:
# ===== 0) Config =====
from pathlib import Path
import os

PROJECT_DIR = Path('')
NOTEBOOK_PATH = PROJECT_DIR / 'exp_backbones_v2.ipynb'

RUN_DIR = PROJECT_DIR / 'runs' / 'skeleton_attention_bilstm 0.77'

BACKBONE = None

THRESHOLD = 0.5
SMOOTH_WIN = 1
FRAME_STEP_OVERRIDE = None
STRIDE = 8

SHOW_INFER_PROGRESS = False
TQDM_MININTERVAL = 1.5

OUT_JSON = PROJECT_DIR / 'full_val_logs' / f"{RUN_DIR.name}_notebook_stable_fullval.json"

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
# ===== 1) Resolve backbone + run stable full-val script =====
import json
import subprocess
import sys

if not RUN_DIR.exists():
    raise FileNotFoundError(f'run_dir not found: {RUN_DIR}')
if not NOTEBOOK_PATH.exists():
    raise FileNotFoundError(f'notebook not found: {NOTEBOOK_PATH}')

backbone_eval = BACKBONE

# 1) try cfg_effective.json
if backbone_eval is None:
    cfg_effective = RUN_DIR / 'cfg_effective.json'
    if cfg_effective.exists():
        try:
            cfg = json.loads(cfg_effective.read_text(encoding='utf-8'))
            backbone_eval = cfg.get('backbone')
        except Exception:
            pass

# 2) fallback: checkpoint cfg
if backbone_eval is None:
    import torch
    best_ckpt = RUN_DIR / 'best.pt'
    if best_ckpt.exists():
        ck = torch.load(str(best_ckpt), map_location='cpu')
        ck_cfg = ck.get('cfg') if isinstance(ck, dict) else None
        if isinstance(ck_cfg, dict):
            backbone_eval = ck_cfg.get('backbone')

if not backbone_eval:
    raise RuntimeError('Could not resolve backbone automatically. Set BACKBONE manually in config cell.')

cmd = [
    sys.executable,
    str(PROJECT_DIR / 'run_full_val_eval_v2.py'),
    '--notebook', str(NOTEBOOK_PATH),
    '--run-dir', str(RUN_DIR),
    '--backbone', str(backbone_eval),
    '--threshold', str(float(THRESHOLD)),
    '--smooth-win', str(int(SMOOTH_WIN)),
    '--out', str(OUT_JSON),
    '--tqdm-mininterval', str(float(TQDM_MININTERVAL)),
]

if FRAME_STEP_OVERRIDE is not None:
    cmd += ['--frame-step', str(int(FRAME_STEP_OVERRIDE))]
if STRIDE is not None:
    cmd += ['--stride', str(int(STRIDE))]
if SHOW_INFER_PROGRESS:
    cmd += ['--show-infer-progress']

print('[full-val] run_dir :', RUN_DIR)
print('[full-val] backbone:', backbone_eval)
print('[full-val] cmd     :', ' '.join(cmd))
print()

subprocess.run(cmd, check=True)

print()
print('[full-val] done. Saved to:', OUT_JSON)


In [ ]:
# ===== 2) Read and show result =====
import json

j = json.loads(OUT_JSON.read_text(encoding='utf-8'))
r = j.get('result', j)

print('=== FULL VAL RESULTS ===')
print(f"macro_f1:        {r.get('macro_f1', float('nan')):.4f}")
print(f"f1 non-violence: {r.get('f1_non_violence', float('nan')):.4f}")
print(f"f1 violence:     {r.get('f1_violence', float('nan')):.4f}")
print('frame_step_used:', j.get('frame_step_used'), f"(source={j.get('frame_step_source')})")
print('stride_used    :', j.get('stride_used'))
print('out_json       :', OUT_JSON)

if 'confusion_matrix_01' in r:
    print()
    print('Confusion matrix [[TN FP],[FN TP]]:')
    print(r['confusion_matrix_01'])

if 'classification_report' in r:
    print()
    print('Classification report:')
    print()
    print(r['classification_report'])
